<a href="https://colab.research.google.com/github/mustah21/SLM-finance/blob/main/group10_ML_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Question Answering Assistant

## Project Overview

This project aims to build a financial question-answering assistant using
the `sweatSmile/FinanceQA` dataset.

The work is divided into three stages:

1. **Fine-tuning:** Adapt a pretrained language model to financial
   question answering using LoRA.
2. **Retrieval-Augmented Generation (RAG):** Build a retrieval component
   that finds relevant financial passages and supplies them to the model.
3. **User Interface:** Build an interface on Hugging Face where users
   can ask questions and view the generated answers.

The fine-tuning stage will provide the model artifacts and inference
instructions needed for the subsequent integration work.

## Scope of This Notebook

This notebook covers the fine-tuning stage, following the Week 4
Qwen2.5 and LoRA activity.

We will use `Qwen/Qwen2.5-1.5B-Instruct` as the starting model and adapt
it using the `sweatSmile/FinanceQA` dataset.

During fine-tuning:
- `CONTEXT` provides the financial passage.
- `QUERY` provides the question.
- `ANSWER` provides the target response.

The model will learn to answer questions using the supplied context.
In the RAG stage, the retrieval component will provide that context.

## Fine-Tuning Objectives

- Prepare the Google Colab environment.
- Load the starting model and tokenizer.
- Prepare FinanceQA examples and reserve data for evaluation.
- Record the model's responses before training.
- Format the training examples using Qwen's chat template.
- Configure and train LoRA adapters.
- Compare responses before and after fine-tuning.
- Export the adapters, tokenizer, and inference instructions.

Evaluation results will be reported after running the experiments.

## 1. Install Dependencies

We install the libraries required to fine-tune Qwen2.5 with LoRA:

- **Transformers:** model loading, tokenization, and training.
- **Datasets:** loading and preparing FinanceQA.
- **PEFT:** configuring and training LoRA adapters.
- **Accelerate:** supporting model execution on the GPU.
- **Hugging Face Hub:** authentication and access to Hub resources.

We use the PyTorch installation provided by Google Colab.

In [ ]:
from huggingface_hub import login, upload_folder
from google.colab import userdata

token = userdata.get('HF_TOKEN')
login(token=token)

# Push your model files
upload_folder(folder_path=".", repo_id="mustafahmad/ML-finance-10", repo_type="model")
# uncomment the above line whenever you wanna push to github
# you will also need a write access token

In [ ]:
# All the packages are now in requirements.txt

%pip install -r requirements.txt

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


### Environment Compatibility Fix

The preinstalled version of `torchao` is incompatible with the installed
PEFT version.

This notebook uses standard LoRA with an FP16 model and does not require
TorchAO quantization. We remove the optional package and restart the
session before loading the model and attaching adapters.

## 2. Verify the Runtime

Fine-tuning requires a CUDA-enabled GPU.

We check the available hardware and record the installed library
versions. We also set a random seed to improve reproducibility,
although results may still vary across hardware and software versions.

In [ ]:
import sys
import torch
from importlib.metadata import version
from transformers import set_seed

# Stop early if the notebook is not connected to a GPU.
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab, select Runtime > "
        "Change runtime type > T4 GPU, then run this cell again."
    )

# Set the random seed for subsequent operations.
SEED = 42
set_seed(SEED)

# Display the available GPU and its total memory.
gpu = torch.cuda.get_device_properties(0)

print(f"GPU: {gpu.name}")
print(f"Total GPU memory: {gpu.total_memory / (1024 ** 3):.2f} GiB")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch CUDA version: {torch.version.cuda}")
print(f"Random seed: {SEED}")

# Record the versions used in this notebook.
print("\nInstalled libraries:")
for package in [
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "huggingface_hub",
]:
    print(f"{package}: {version(package)}")

GPU: Tesla T4
Total GPU memory: 14.56 GiB
Python version: 3.13.15
PyTorch version: 2.11.0+cu128
PyTorch CUDA version: 12.8
Random seed: 42

Installed libraries:
transformers: 5.17.0
datasets: 5.0.1
peft: 0.21.0
accelerate: 1.15.0
huggingface_hub: 1.33.0


## 3. Authenticate with Hugging Face

FinanceQA is a gated dataset. The Hugging Face account used in this
notebook must have permission to access it.

We retrieve the token from Colab Secrets and authenticate with the Hub.
The token is not included in the notebook code or printed in the output.

Successful authentication confirms the account credentials; dataset
access will be verified when the data is downloaded.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token from Colab's secure secret storage.
hf_token = userdata.get("HF_TOKEN")

# Authenticate without adding the token to Git credentials.
login(token=hf_token, add_to_git_credential=False)

# Remove this reference to the token from the notebook variables.
del hf_token

print("Hugging Face authentication completed.")
print("Dataset access will be checked when FinanceQA is loaded.")

Hugging Face authentication completed.
Dataset access will be checked when FinanceQA is loaded.


## 4. Load the Starting Model and Tokenizer

We use `Qwen/Qwen2.5-1.5B-Instruct`, following the model choice in the
Week 4 fine-tuning lab.

The tokenizer converts text into token IDs and provides the chat template
used to format conversations.

We load the model on the GPU in FP16 to reduce memory usage.
We preserve the tokenizer's existing padding token when available.

At this stage, the model has not been fine-tuned on FinanceQA.
LoRA adapters will be added after recording the baseline results.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Load the tokenizer and preserve its existing padding token.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# Load the model in FP16 on the Colab GPU.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="cuda",
)

# Align the model's padding configuration with the tokenizer.
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# Disable caching in preparation for training.
# We can enable it explicitly during baseline generation.
model.config.use_cache = False

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {model.device}")
print(f"Model dtype: {model.dtype}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Padding token: {tokenizer.pad_token!r}")
print(f"End-of-sequence token: {tokenizer.eos_token!r}")
print(
    "GPU memory allocated by PyTorch: "
    f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GiB"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Model dtype: torch.float16
Parameters: 1,543,714,304
Padding token: '<|endoftext|>'
End-of-sequence token: '<|im_end|>'
GPU memory allocated by PyTorch: 2.88 GiB


## 5. Load and Inspect FinanceQA

We load FinanceQA from the Hugging Face Hub using the authenticated account.

Before preprocessing, we inspect:
- The available dataset splits.
- The number of examples in each split.
- The column names and data types.
- One example from each non-empty split.

The dataset card describes `CONTEXT`, `QUERY`, and `ANSWER` as the fields
needed for question answering. We verify the actual schema before
building the training examples.

No filtering, splitting, or tokenization is performed in this step.

In [ ]:
from datasets import load_dataset
from pprint import pprint

DATASET_NAME = "sweatSmile/FinanceQA"

# Download the dataset using the authenticated Hugging Face account.
raw_data = load_dataset(DATASET_NAME, token=True)

print("Dataset structure:")
print(raw_data)

# Inspect the actual schema and one example from each split.
for split_name, split_data in raw_data.items():
    print(f"\nSplit: {split_name}")
    print(f"Number of examples: {len(split_data):,}")
    print(f"Columns: {split_data.column_names}")
    print("Features:")
    pprint(split_data.features)

    if len(split_data) > 0:
        print("\nFirst example:")
        pprint(split_data[0], sort_dicts=False)

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  441kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  112kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3705 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/927 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT', '__index_level_0__'],
        num_rows: 3705
    })
    test: Dataset({
        features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT', '__index_level_0__'],
        num_rows: 927
    })
})

Split: train
Number of examples: 3,705
Columns: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT', '__index_level_0__']
Features:
{'ANSWER': Value('string'),
 'COMPANY_ID': Value('string'),
 'CONTEXT': Value('string'),
 'QUERY': Value('string'),
 '__index_level_0__': Value('int64')}

First example:
{'COMPANY_ID': 'ARCOTECH_2023_converted.txt_0',
 'QUERY': 'What is the equity share capital of the company?',
 'ANSWER': 'The equity share capital of the company is 21.',
 'CONTEXT': 'Symbol: ARCOTECH Company Name: Arcotech Ltd. EQUITIES AND '
            "LIABILITIES: nan SHAREHOLDER'S FUNDS: nan Equity Share Capital: "
            '21 Total Share Capital: 24.32 Reserves and Surplus: -113.53 Tot

## 6. Clean the Data and Create the Validation Split

We keep the original test split separate from model development.

Before creating the validation split, we:
- Remove examples with missing or empty required text fields.
- Exclude training inputs that also appear in the test split.
- Remove repeated training examples with identical inputs and answers.
- Exclude training inputs associated with conflicting reference answers.

An input is identified by its question and context, with whitespace
normalized for comparison.

We then reserve 20% of the remaining training examples for validation,
using a fixed random seed.

These checks prevent identical inputs from appearing across splits.
They do not guarantee that companies or source documents are disjoint.

In [ ]:
from datasets import DatasetDict

VALIDATION_SIZE = 0.20
columns_to_keep = ["COMPANY_ID", "QUERY", "ANSWER", "CONTEXT"]


def normalize_text(text):
    """Normalize whitespace without changing numbers or punctuation."""
    return " ".join(text.split())


def input_key(example):
    """Identify an input using its question and financial context."""
    return (
        normalize_text(example["QUERY"]),
        normalize_text(example["CONTEXT"]),
    )


def has_required_text(example):
    """Keep examples with non-empty questions, answers, and contexts."""
    return all(
        isinstance(example[field], str) and bool(example[field].strip())
        for field in ["QUERY", "ANSWER", "CONTEXT"]
    )


# Start again from the original dataset.
clean_data = DatasetDict()

for split_name in ["train", "test"]:
    selected = raw_data[split_name].select_columns(columns_to_keep)
    clean_data[split_name] = selected.filter(has_required_text)

    print(
        f"{split_name}: "
        f"{len(selected) - len(clean_data[split_name])} "
        "incomplete examples removed."
    )

# Preserve the valid examples from the original test split.
test_data = clean_data["test"]
test_keys = {input_key(row) for row in test_data}

# Group training rows by their normalized input.
groups = {}

for index, row in enumerate(clean_data["train"]):
    key = input_key(row)

    if key not in groups:
        groups[key] = {
            "first_index": index,
            "answers": set(),
            "row_count": 0,
        }

    groups[key]["answers"].add(normalize_text(row["ANSWER"]))
    groups[key]["row_count"] += 1

# Keep one example per input, excluding test overlap and conflicting labels.
keep_indices = []
overlap_rows = 0
conflicting_rows = 0
duplicate_rows = 0

for key, group in groups.items():
    if key in test_keys:
        overlap_rows += group["row_count"]
    elif len(group["answers"]) > 1:
        conflicting_rows += group["row_count"]
    else:
        keep_indices.append(group["first_index"])
        duplicate_rows += group["row_count"] - 1

training_pool = clean_data["train"].select(sorted(keep_indices))

print(f"\nTraining rows overlapping test removed: {overlap_rows}")
print(f"Training rows with conflicting answers removed: {conflicting_rows}")
print(f"Additional duplicate training rows removed: {duplicate_rows}")
print(f"Examples available for train/validation: {len(training_pool)}")

if len(training_pool) < 2:
    raise ValueError("Not enough examples remain to create both splits.")

# Reserve 20% for validation after cleaning and deduplication.
split = training_pool.train_test_split(
    test_size=VALIDATION_SIZE,
    seed=SEED,
)

dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": test_data,
})

# Verify that no identical inputs appear across splits.
keys = {
    name: {input_key(row) for row in data}
    for name, data in dataset.items()
}

for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = len(keys[left] & keys[right])
    print(f"Identical inputs in {left} / {right}: {overlap}")
    assert overlap == 0, f"Unexpected overlap between {left} and {right}."

print("\nFinal dataset:")
print(dataset)

Filter:   0%|          | 0/3705 [00:00<?, ? examples/s]

train: 0 incomplete examples removed.


Filter:   0%|          | 0/927 [00:00<?, ? examples/s]

test: 0 incomplete examples removed.

Training rows overlapping test removed: 6
Training rows with conflicting answers removed: 0
Additional duplicate training rows removed: 0
Examples available for train/validation: 3699
Identical inputs in train / validation: 0
Identical inputs in train / test: 0
Identical inputs in validation / test: 0

Final dataset:
DatasetDict({
    train: Dataset({
        features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT'],
        num_rows: 2959
    })
    validation: Dataset({
        features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT'],
        num_rows: 740
    })
    test: Dataset({
        features: ['COMPANY_ID', 'QUERY', 'ANSWER', 'CONTEXT'],
        num_rows: 927
    })
})


### Data Preparation Results

Six training examples were removed because their question-context pairs
also appeared in the original test split.

The remaining 3,699 examples were divided into:
- Training: 2,959 examples.
- Validation: 740 examples.

The original test split contains 927 examples.

No identical question-context pairs remain across splits. However,
different examples may still refer to the same company or source document.

## 7. Baseline Inference Before Fine-Tuning

We record the starting model's responses to three validation examples.

Each input contains a financial passage and a question. The reference
answer is kept separate and is never included in the inference prompt.

We use Qwen's chat template and greedy decoding. The same examples,
prompt format, and generation settings will be used after fine-tuning.

This small sample provides a qualitative comparison. It is not a
complete performance evaluation, and improvement is not assumed.

In [ ]:
import json
import torch

SYSTEM_PROMPT = (
    "You are a financial question-answering assistant. "
    "Answer the question using only the provided context. "
    "Give a concise answer and preserve the numbers and units "
    "stated in the context. Do not invent missing information."
)

MAX_NEW_TOKENS = 128


def build_messages(context, question, answer=None):
    """Use one consistent message structure for training and inference."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}",
        },
    ]

    # Include the reference answer only when preparing training examples.
    if answer is not None:
        messages.append({"role": "assistant", "content": answer})

    return messages


def generate_answer(context, question):
    """Generate an answer without exposing the reference answer."""
    messages = build_messages(context, question)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    model.eval()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens.
    generated_ids = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()


# Select a reproducible sample from validation, leaving test for later.
baseline_examples = dataset["validation"].shuffle(seed=SEED).select(
    range(min(3, len(dataset["validation"])))
)

baseline_results = []

for number, example in enumerate(baseline_examples, start=1):
    prediction = generate_answer(example["CONTEXT"], example["QUERY"])

    baseline_results.append({
        "company_id": example["COMPANY_ID"],
        "context": example["CONTEXT"],
        "question": example["QUERY"],
        "reference_answer": example["ANSWER"],
        "baseline_answer": prediction,
    })

    print(f"\nExample {number}")
    print("Question:", example["QUERY"])
    print("Reference answer:", example["ANSWER"])
    print("Starting model answer:", prediction)

# Save the responses for the before-and-after comparison.
with open("/content/financeqa_baseline.json", "w", encoding="utf-8") as file:
    json.dump(baseline_results, file, ensure_ascii=False, indent=2)

print("\nBaseline saved to /content/financeqa_baseline.json")


Example 1
Question: What is the total reserves and surplus of the company?
Reference answer: The total reserves and surplus of the company are 854.08.
Starting model answer: The total reserves and surplus of the company is 854.08.

Example 2
Question: What is the equity share capital of the company?
Reference answer: The equity share capital of the company is 50.35.
Starting model answer: The equity share capital of the company is £50.35.

Example 3
Question: What is the total value of assets of the company?
Reference answer: 377.9
Starting model answer: The total value of assets for the company, as shown in the given data, is 377.9.

Baseline saved to /content/financeqa_baseline.json


### Baseline Observations

The starting model reproduced the reference numerical values in all
three validation examples.

Examples 1 and 3 differ from the reference answers in wording.
In Example 2, the model added a pound sign (£) that is absent from the
reference answer. The source context must be checked to determine
whether this currency is supported.

These examples suggest that the starting model can already extract
some financial values from the supplied context. Fine-tuning will be
evaluated for its effect on answer accuracy, conciseness, and adherence
to the information provided.

Three examples are insufficient to establish overall performance.
Improvement after fine-tuning is not assumed.

## 8. Format and Tokenize the Training Examples

We reuse the message structure defined for baseline inference.

Each training conversation contains:
- The system instruction.
- The financial context and question.
- The reference answer as the assistant's response.

We apply Qwen's chat template to convert each complete conversation
into token IDs. During training, `add_generation_prompt=False` because
the assistant's response is already included.

We inspect sequence lengths before training. No text is truncated at
this stage, so financial information and target answers remain intact.

Padding will be applied dynamically when training batches are created.
The test split remains reserved for final evaluation.

In [ ]:
from datasets import DatasetDict
from statistics import median

# Initial sequence-length budget for training on the T4.
MAX_LENGTH = 1024


def tokenize_example(example):
    """Tokenize a complete conversation, including the target answer."""
    messages = build_messages(
        context=example["CONTEXT"],
        question=example["QUERY"],
        answer=example["ANSWER"],
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        truncation=False,
        padding=False,
        return_dict=True,
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }


# Prepare only the training and validation splits.
tokenized_data = DatasetDict({
    split_name: dataset[split_name].map(
        tokenize_example,
        remove_columns=dataset[split_name].column_names,
        desc=f"Tokenizing {split_name}",
    )
    for split_name in ["train", "validation"]
})

# Check sequence lengths before applying any truncation.
over_limit = 0

for split_name, split_data in tokenized_data.items():
    lengths = [len(ids) for ids in split_data["input_ids"]]
    count = sum(length > MAX_LENGTH for length in lengths)
    over_limit += count

    print(f"\n{split_name}:")
    print(f"Examples: {len(lengths):,}")
    print(f"Shortest sequence: {min(lengths)} tokens")
    print(f"Median sequence: {median(lengths):.0f} tokens")
    print(f"Longest sequence: {max(lengths)} tokens")
    print(f"Sequences exceeding {MAX_LENGTH} tokens: {count}")

if over_limit:
    raise ValueError(
        "Some sequences exceed the initial training length budget. "
        "Review the length statistics before continuing. "
        "No examples have been truncated or removed."
    )

print("\nTokenization completed. All sequences fit the length budget.")

Tokenizing train:   0%|          | 0/2959 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/740 [00:00<?, ? examples/s]


train:
Examples: 2,959
Shortest sequence: 164 tokens
Median sequence: 206 tokens
Longest sequence: 289 tokens
Sequences exceeding 1024 tokens: 0

validation:
Examples: 740
Shortest sequence: 181 tokens
Median sequence: 206 tokens
Longest sequence: 286 tokens
Sequences exceeding 1024 tokens: 0

Tokenization completed. All sequences fit the length budget.


## 9. Configure LoRA Adapters

We use LoRA to adapt the model while keeping its original parameters
frozen.

Following the Week 4 lab, we configure:
- Rank (`r`): 16.
- Scaling factor (`lora_alpha`): 32.
- Target modules: attention and feed-forward projection layers.
- Bias training: disabled.

Only the adapter parameters will be updated during training.

We print the number of trainable parameters to verify that the model
is configured for parameter-efficient fine-tuning.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import set_seed

# Make adapter initialization reproducible.
set_seed(SEED)

# Configure the LoRA adapters following the lab.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

# Attach trainable adapters and freeze the original model parameters.
model = get_peft_model(model, lora_config)

# Keep caching disabled for the upcoming training stage.
model.config.use_cache = False

# Verify the number and proportion of trainable parameters.
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 10. Configure a Short Training Run

To fit the available Colab resources, we run one training epoch over
all 2,959 training examples.

We use a batch size of 4 and no gradient accumulation. This produces
approximately 740 optimizer steps on one GPU if the batch fits in memory.

Automatic batch-size reduction is enabled in case training runs out
of GPU memory. A smaller batch will increase the number of steps.

FP16 and gradient checkpointing remain enabled. Validation and checkpoint
saving take place at the end of the epoch.

This is an initial experiment with a limited compute budget. We will
evaluate its results before deciding whether more training is useful.

In [ ]:
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

# Use a separate directory for this shorter experiment.
RUN_DIR = "/content/financeqa_training_1epoch"
FINAL_DIR = "/content/financeqa_qwen_lora"

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=RUN_DIR,

    # Run one complete pass through the training dataset.
    num_train_epochs=1,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    auto_find_batch_size=True,
    lr_scheduler_type="cosine",

    # Keep memory-saving settings for the T4.
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Evaluate and save at the end of the epoch.
    per_device_eval_batch_size=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,

    logging_steps=25,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    optim="adamw_torch",
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("One-epoch training configured.")
print("Initial batch size: 4")
print("Gradient accumulation: 1")
print("Expected optimizer steps with batch size 4: 740")

One-epoch training configured.
Initial batch size: 4
Gradient accumulation: 1
Expected optimizer steps with batch size 4: 740


## 11. Run Training and Save the Selected Model

We start training and automatically save the selected LoRA adapters
and tokenizer after training finishes.

We also save the training history, baseline responses, inference
configuration, and package versions for later comparison and reuse.

The exported adapter requires the original
`Qwen/Qwen2.5-1.5B-Instruct` model for inference.

In [ ]:
import json
from pathlib import Path
from importlib.metadata import version

# Train the adapters. The best checkpoint is restored automatically.
train_result = trainer.train()

# Save the selected adapters and tokenizer.
export_dir = Path(FINAL_DIR)
export_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
trainer.save_metrics("train", train_result.metrics)
trainer.state.save_to_json(str(export_dir / "trainer_state.json"))

# Preserve the baseline comparison.
with open(export_dir / "baseline_results.json", "w", encoding="utf-8") as file:
    json.dump(baseline_results, file, ensure_ascii=False, indent=2)

# Save the prompt and generation settings needed by the team.
inference_config = {
    "base_model": MODEL_NAME,
    "system_prompt": SYSTEM_PROMPT,
    "user_template": "Context:\n{context}\n\nQuestion:\n{question}",
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "seed": SEED,
    "split_sizes": {
        name: len(data) for name, data in dataset.items()
    },
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_loss": trainer.state.best_metric,
}

with open(export_dir / "inference_config.json", "w", encoding="utf-8") as file:
    json.dump(inference_config, file, ensure_ascii=False, indent=2)

# Record the main package versions used in this run.
packages = [
    "torch", "transformers", "datasets",
    "peft", "accelerate", "huggingface_hub",
]

with open(export_dir / "package_versions.txt", "w", encoding="utf-8") as file:
    for package in packages:
        file.write(f"{package}=={version(package)}\n")

print("\nTraining completed.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)
print("Adapters and tokenizer saved to:", FINAL_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,0.336381,0.334642



Training completed.
Best checkpoint: /content/financeqa_training_1epoch/checkpoint-740
Best validation loss: 0.3346417248249054
Adapters and tokenizer saved to: /content/financeqa_qwen_lora


In [ ]:
import shutil
from google.colab import files

# Download the adapter package before the Colab runtime is deleted.
archive_path = shutil.make_archive(
    "/content/financeqa_qwen_lora",
    "zip",
    root_dir=FINAL_DIR,
)

files.download(archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Training Results

The model completed one training epoch with 740 optimizer steps.

- Last logged training loss: 0.331757.
- Validation loss: 0.329921.
- Selected checkpoint: checkpoint-740.

The LoRA adapters and tokenizer were saved successfully.

These losses measure next-token prediction over complete conversations.
They are not answer-accuracy scores. Generated answers must be evaluated
separately to assess the effect of fine-tuning.

## 13. Compare Baseline and Fine-Tuned Responses

We evaluate the fine-tuned model on the same three validation examples
used before training.

The system prompt, question, context, and generation settings remain
unchanged. We compare the saved baseline responses with newly generated
responses from the fine-tuned model.

We inspect numerical correctness, supported units, and answer wording.
This is a qualitative check; final test-set evaluation follows separately.

In [ ]:
import json
import textwrap
from pathlib import Path

# Use the selected model already loaded by Trainer.
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

comparison_results = []

for number, baseline in enumerate(baseline_results, start=1):
    # Reuse exactly the same context, question, and generation function.
    fine_tuned_answer = generate_answer(
        context=baseline["context"],
        question=baseline["question"],
    )

    result = {
        **baseline,
        "fine_tuned_answer": fine_tuned_answer,
    }
    comparison_results.append(result)

    print(f"\nEXAMPLE {number}")

    # Wrap long responses so they remain visible in Colab.
    for label, key in [
        ("Question", "question"),
        ("Reference answer", "reference_answer"),
        ("Before fine-tuning", "baseline_answer"),
        ("After fine-tuning", "fine_tuned_answer"),
    ]:
        print(textwrap.fill(
            f"{label}: {result[key]}",
            width=90,
            subsequent_indent="    ",
        ))

# Save the comparison alongside the adapter files.
comparison_path = Path(FINAL_DIR) / "before_after_comparison.json"

with open(comparison_path, "w", encoding="utf-8") as file:
    json.dump(comparison_results, file, ensure_ascii=False, indent=2)

print(f"\nComparison saved to: {comparison_path}")


EXAMPLE 1
Question: What is the total reserves and surplus of the company?
Reference answer: The total reserves and surplus of the company are 854.08.
Before fine-tuning: The total reserves and surplus of the company is 854.08.
After fine-tuning: The total reserves and surplus of the company is 854.08.

EXAMPLE 2
Question: What is the equity share capital of the company?
Reference answer: The equity share capital of the company is 50.35.
Before fine-tuning: The equity share capital of the company is £50.35.
After fine-tuning: The equity share capital of the company is 50.35.

EXAMPLE 3
Question: What is the total value of assets of the company?
Reference answer: 377.9
Before fine-tuning: The total value of assets for the company, as shown in the given data,
    is 377.9.
After fine-tuning: The total value of assets of the company is $377.9 million.

Comparison saved to: /content/financeqa_qwen_lora/before_after_comparison.json


### Preliminary Interpretation

The starting model already reproduced the reference numerical values
in all three examples, leaving limited room for improvement on these
specific questions.

Fine-tuning preserved the numerical values. It removed the pound sign
in Example 2 but introduced a dollar sign in Example 3. Neither symbol
appears in the corresponding reference answer; the source contexts
must be checked to determine whether these units are supported.

These observations do not establish an overall improvement.
A comparison on the held-out test set is needed before drawing conclusions.

## 14. Evaluate on a Held-Out Test Sample

We compare the original model and the fine-tuned model on the same
100 test examples, selected with a fixed random seed.

The original model is evaluated by temporarily disabling the LoRA
adapters. Both models receive identical contexts, questions, and
generation settings.

We report:
- Normalized exact match with the reference answer.
- Numerical agreement: whether the extracted numbers match the
  reference numbers, including repeated occurrences.
- The number of responses containing currency symbols absent from
  the reference answer.

These are diagnostic measures. Exact match penalizes valid paraphrases.
Numerical agreement does not verify units or the meaning of a response.
Additional currency symbols require manual inspection of the context;
they are not automatically classified as errors.

This evaluation covers a sample of the test split. It does not establish
performance on entirely unseen companies or documents.

In [ ]:
import json
import re
from collections import Counter
from decimal import Decimal
from pathlib import Path
from tqdm.auto import tqdm

# Select the same test examples for both models.
TEST_SAMPLE_SIZE = 100

test_sample = dataset["test"].shuffle(seed=SEED).select(
    range(min(TEST_SAMPLE_SIZE, len(dataset["test"])))
)

model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()


def normalize_answer(text):
    """Normalize case and whitespace, preserving punctuation and units."""
    return " ".join(text.casefold().split())


def extract_numbers(text):
    """Extract decimal numbers using English-style comma grouping."""
    matches = re.findall(r"[+-]?\d+(?:,\d{3})*(?:\.\d+)?", text)

    return Counter(
        Decimal(number.replace(",", ""))
        for number in matches
    )


def additional_currency_symbols(prediction, reference):
    """Flag symbols absent from the reference, without judging correctness."""
    symbols = set("$€£¥₹")
    return sorted(
        (set(prediction) & symbols) - (set(reference) & symbols)
    )


test_results = []

for sample_index, example in enumerate(
    tqdm(test_sample, desc="Comparing original and fine-tuned models")
):
    # Disable adapters to obtain a response from the frozen original model.
    with model.disable_adapter():
        original_answer = generate_answer(
            example["CONTEXT"],
            example["QUERY"],
        )

    # The context manager restores the adapters automatically.
    fine_tuned_answer = generate_answer(
        example["CONTEXT"],
        example["QUERY"],
    )

    test_results.append({
        "sample_index": sample_index,
        "company_id": example["COMPANY_ID"],
        "context": example["CONTEXT"],
        "question": example["QUERY"],
        "reference_answer": example["ANSWER"],
        "original_answer": original_answer,
        "fine_tuned_answer": fine_tuned_answer,
    })


def summarize_results(answer_field):
    exact_matches = 0
    numerical_matches = 0
    numerical_examples = 0
    currency_flags = 0

    for result in test_results:
        prediction = result[answer_field]
        reference = result["reference_answer"]

        exact_matches += (
            normalize_answer(prediction) == normalize_answer(reference)
        )

        reference_numbers = extract_numbers(reference)

        # Only score numerical agreement when the reference has numbers.
        if reference_numbers:
            numerical_examples += 1
            numerical_matches += (
                extract_numbers(prediction) == reference_numbers
            )

        currency_flags += bool(
            additional_currency_symbols(prediction, reference)
        )

    count = len(test_results)

    return {
        "evaluated_examples": count,
        "normalized_exact_match_percent": round(
            100 * exact_matches / count, 2
        ),
        "references_containing_numbers": numerical_examples,
        "numerical_agreement_percent": (
            round(100 * numerical_matches / numerical_examples, 2)
            if numerical_examples else None
        ),
        "responses_flagged_for_additional_currency_symbols": currency_flags,
    }


test_metrics = {
    "sample_size": len(test_results),
    "total_test_examples": len(dataset["test"]),
    "seed": SEED,
    "original_model": summarize_results("original_answer"),
    "fine_tuned_model": summarize_results("fine_tuned_answer"),
}

# Save predictions and metrics with the model package.
export_dir = Path(FINAL_DIR)
export_dir.mkdir(parents=True, exist_ok=True)

for filename, content in [
    ("test_predictions.json", test_results),
    ("test_metrics.json", test_metrics),
]:
    with open(export_dir / filename, "w", encoding="utf-8") as file:
        json.dump(content, file, ensure_ascii=False, indent=2)

print("\nTEST EVALUATION RESULTS")
print(json.dumps(test_metrics, indent=2))
print("\nPredictions and metrics saved in:", FINAL_DIR)

Comparing original and fine-tuned models:   0%|          | 0/100 [00:00<?, ?it/s]


TEST EVALUATION RESULTS
{
  "sample_size": 100,
  "total_test_examples": 927,
  "seed": 42,
  "original_model": {
    "evaluated_examples": 100,
    "normalized_exact_match_percent": 27.0,
    "references_containing_numbers": 95,
    "numerical_agreement_percent": 94.74,
    "responses_flagged_for_additional_currency_symbols": 19
  },
  "fine_tuned_model": {
    "evaluated_examples": 100,
    "normalized_exact_match_percent": 57.0,
    "references_containing_numbers": 95,
    "numerical_agreement_percent": 95.79,
    "responses_flagged_for_additional_currency_symbols": 11
  }
}

Predictions and metrics saved in: /content/financeqa_qwen_lora


In [ ]:
import textwrap

flagged_results = [
    result for result in test_results
    if additional_currency_symbols(
        result["fine_tuned_answer"],
        result["reference_answer"],
    )
]

print(f"Fine-tuned responses requiring currency review: {len(flagged_results)}")

for result in flagged_results[:3]:
    print(f"\nTEST SAMPLE INDEX: {result['sample_index']}")

    for label, key in [
        ("Question", "question"),
        ("Context", "context"),
        ("Reference", "reference_answer"),
        ("Original model", "original_answer"),
        ("Fine-tuned model", "fine_tuned_answer"),
    ]:
        print(textwrap.fill(
            f"{label}: {result[key]}",
            width=90,
            subsequent_indent="    ",
        ))

Fine-tuned responses requiring currency review: 11

TEST SAMPLE INDEX: 10
Question: What is the total value of assets of the company?
Context: 1.4 Total Current Liabilities: 132 Total Capital And Liabilities: 893.4 ASSETS:
    nan NON-CURRENT ASSETS: nan Tangible Assets: 102.31 Intangible Assets: 0 Capital Work-
    In-Progress: 0 Other Assets: 0 Fixed Assets: 102.31 Non-Current Investments: 216.69
    Deferred Tax Assets [Net]: 6.76 Long Term Loans And Advances: 0 Other Non-Current
    Assets: 4.19 Total
Reference: The total value of assets of the company is 893.4.
Original model: The total value of assets for the company is $893.4.
Fine-tuned model: The total value of assets of the company is $893.4 million.

TEST SAMPLE INDEX: 12
Question: What is the total value of assets of the company?
Context: 125.75 Total Capital And Liabilities: 852.14 ASSETS: nan NON-CURRENT ASSETS: nan
    Tangible Assets: 470 Intangible Assets: 0.11 Capital Work-In-Progress: 0 Other Assets:
    0 Fixed Asse

## 16. Results and Conclusions

We evaluated the original and fine-tuned models on the same fixed
sample of 100 test examples.

Normalized exact match increased from 27% to 57%.

Among the 95 examples whose reference answers contained numbers,
numerical agreement increased from 90/95 (94.74%) to 91/95 (95.79%).
This represents one additional numerical match.

The number of responses flagged for currency symbols absent from
the reference decreased from 19 to 10.

Overall, fine-tuning improved agreement with the reference wording
on this sample. The improvement in numerical agreement was small.

### Remaining Limitations

Manual inspection identified unsupported currency and scale additions.
For example, some responses introduced "$" or "million" even though
neither appeared in the supplied context.

Our numerical metric does not capture these semantic errors.
Matching the digits alone does not guarantee a correct financial answer.

The evaluation covered 100 of 927 test examples. Results may not
generalize to the full test set or to unseen companies and documents.
No claim of statistical significance is made.

### Handoff

The deliverable contains the LoRA adapters, tokenizer, inference
configuration, evaluation results, and a reusable inference script.

The RAG component should supply the relevant financial context.
The interface can call the inference script with a question and context.

The original Qwen2.5-1.5B-Instruct model is required to use the adapters.

In [ ]:
%%writefile /content/financeqa_qwen_lora/inference.py
"""Load the FinanceQA adapter and answer questions using supplied context."""

import json
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftConfig, PeftModel


class FinanceAssistant:
    def __init__(self, adapter_dir=None):
        # Default to the folder containing this script.
        directory = (
            Path(adapter_dir)
            if adapter_dir is not None
            else Path(__file__).resolve().parent
        )

        with open(directory / "inference_config.json", encoding="utf-8") as file:
            self.settings = json.load(file)

        adapter_config = PeftConfig.from_pretrained(str(directory))

        # Use FP16 on CUDA and FP32 when running on CPU.
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32

        self.tokenizer = AutoTokenizer.from_pretrained(str(directory))

        base_model = AutoModelForCausalLM.from_pretrained(
            adapter_config.base_model_name_or_path,
            dtype=dtype,
            device_map=device,
        )

        self.model = PeftModel.from_pretrained(
            base_model,
            str(directory),
            is_trainable=False,
        )

        self.model.config.use_cache = True
        self.model.eval()

    def answer(self, question, context):
        """Return an answer using the same prompt as the training notebook."""
        if not isinstance(question, str) or not question.strip():
            raise ValueError("Provide a non-empty question.")

        if not isinstance(context, str) or not context.strip():
            raise ValueError("Provide a non-empty financial context.")

        messages = [
            {
                "role": "system",
                "content": self.settings["system_prompt"],
            },
            {
                "role": "user",
                "content": self.settings["user_template"].format(
                    context=context,
                    question=question,
                ),
            },
        ]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)

        input_length = inputs["input_ids"].shape[1]
        input_budget = self.settings.get("max_input_tokens", 1024)

        # Ask the retrieval component to shorten oversized input.
        # Do not silently discard potentially relevant financial information.
        if input_length > input_budget:
            raise ValueError(
                f"Input contains {input_length} tokens; "
                f"the application budget is {input_budget}. "
                "Supply a shorter relevant context."
            )

        with torch.inference_mode():
            output = self.model.generate(
                **inputs,
                max_new_tokens=self.settings["max_new_tokens"],
                do_sample=self.settings["do_sample"],
                use_cache=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        generated_ids = output[0, input_length:]
        return self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

Writing /content/financeqa_qwen_lora/inference.py


In [ ]:
import json
from pathlib import Path
from importlib.metadata import version

export_dir = Path(FINAL_DIR)

# Preserve the actual training arguments.
(export_dir / "training_config.json").write_text(
    trainer.args.to_json_string(),
    encoding="utf-8",
)

# Record the input budget used by the reusable inference script.
config_path = export_dir / "inference_config.json"

with open(config_path, encoding="utf-8") as file:
    inference_settings = json.load(file)

inference_settings["max_input_tokens"] = MAX_LENGTH

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(inference_settings, file, ensure_ascii=False, indent=2)

# Record inference dependencies. Install an appropriate PyTorch build separately.
runtime_packages = ["transformers", "peft", "accelerate", "huggingface_hub"]

(export_dir / "requirements.txt").write_text(
    "\n".join(
        f"{package}=={version(package)}"
        for package in runtime_packages
    ) + "\n",
    encoding="utf-8",
)

readme = """# FinanceQA LoRA Adapter

## Model

- Base model: Qwen/Qwen2.5-1.5B-Instruct.
- Dataset: sweatSmile/FinanceQA.
- Training: one epoch, 740 optimizer steps.
- LoRA rank: 16; alpha: 32.
- Training examples: 2,959.
- Validation examples: 740.
- Final validation loss: approximately 0.329921.

This folder contains adapters, not the complete base model.
The inference script downloads the base model when it is not cached.

## Setup

Use a Python environment with a compatible PyTorch installation.
The experiment used Google Colab, Python 3.13, and a Tesla T4.
See package_versions.txt for the recorded environment.

Install the inference dependencies:

    pip install -r requirements.txt

TorchAO is not required. An incompatible preinstalled version caused
a PEFT error in Colab and was removed with:

    pip uninstall -y torchao

Restart Python after changing already imported packages.

## Usage

Extract the package and run this code from its directory:

    from inference import FinanceAssistant

    assistant = FinanceAssistant()

    # Synthetic example illustrating the API:
    response = assistant.answer(
        question="What is the total value of assets?",
        context="Total assets: 120."
    )
    print(response)

Create the assistant once and reuse it for subsequent questions.
CUDA is recommended; CPU inference uses FP32 and is slower.

## RAG and UI Integration

The RAG component supplies the relevant passage as context.
The UI supplies the user's question and displays the returned string.

The script does not retrieve documents or verify factual correctness.
It preserves the notebook's system prompt and generation settings.

Inputs exceeding 1,024 tokens after chat formatting are rejected.
Training conversations were at most 289 tokens long; longer retrieved
contexts have not been evaluated in this experiment.

## Test Sample Results

Evaluation used 100 of 927 test examples, with seed 42.

- Normalized exact match: 27% original, 57% fine-tuned.
- Numerical agreement: 90/95 original, 91/95 fine-tuned.
- Additional currency-symbol flags: 19 original, 10 fine-tuned.

Numerical agreement is not overall factual accuracy.
Manual review found unsupported currency and scale additions,
including "$" and "million".

Results are preliminary and do not establish performance on
unseen companies or documents.

See test_metrics.json and test_predictions.json for the evaluation.
"""

(export_dir / "HANDOFF.md").write_text(readme, encoding="utf-8")

print("Inference instructions and experiment configuration saved.")

Inference instructions and experiment configuration saved.


## 19. Verify the Saved Adapter Package

We release the training model from memory and load a new model instance
using the exported adapter and inference script.

We repeat one previously evaluated input and compare its output with
the saved prediction. This checks that the exported package reproduces
the notebook's inference behavior for that example.

In [ ]:
import gc
import importlib.util
import json
from pathlib import Path
import torch

# Preserve an input and its expected output before releasing the model.
reload_example = test_results[0]
expected_answer = reload_example["fine_tuned_answer"]

# Release the training model and optimizer references.
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

# Import the inference script from the exported package.
script_path = Path(FINAL_DIR) / "inference.py"
spec = importlib.util.spec_from_file_location("financeqa_inference", script_path)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

# Load the exported adapter through the interface intended for teammates.
assistant = inference_module.FinanceAssistant(FINAL_DIR)

reloaded_answer = assistant.answer(
    question=reload_example["question"],
    context=reload_example["context"],
)

reload_check = {
    "sample_index": reload_example["sample_index"],
    "expected_answer": expected_answer,
    "reloaded_answer": reloaded_answer,
    "matches": reloaded_answer == expected_answer,
}

with open(
    Path(FINAL_DIR) / "reload_check.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(reload_check, file, ensure_ascii=False, indent=2)

print(json.dumps(reload_check, ensure_ascii=False, indent=2))

if not reload_check["matches"]:
    raise RuntimeError(
        "The reloaded answer differs. Review the outputs before final delivery."
    )

print("\nReload test passed. The saved package reproduced the answer.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{
  "sample_index": 0,
  "expected_answer": "The expenditure in foreign currency of the company is 36.6.",
  "reloaded_answer": "The expenditure in foreign currency of the company is 36.6.",
  "matches": true
}

Reload test passed. The saved package reproduced the answer.


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "inference.py",
    "inference_config.json",
    "training_config.json",
    "requirements.txt",
    "package_versions.txt",
    "HANDOFF.md",
    "test_metrics.json",
    "test_predictions.json",
    "before_after_comparison.json",
    "reload_check.json",
]

missing = [
    name for name in required_files
    if not (Path(FINAL_DIR) / name).exists()
]

if missing:
    raise FileNotFoundError(f"Missing delivery files: {missing}")

if not reload_check["matches"]:
    raise RuntimeError("The reload check must pass before final delivery.")

archive_path = shutil.make_archive(
    "/content/financeqa_finetuning_final",
    "zip",
    root_dir=FINAL_DIR,
)

print("Final package:", archive_path)
files.download(archive_path)

Final package: /content/financeqa_finetuning_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>